# Limpeza e Pré-processamento dos Dados

Este notebook realiza o tratamento dos valores ausentes, análise de outliers, codificação das variáveis categóricas e padronização das variáveis numéricas.

## Carregamento do dataset

In [14]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

df = pd.read_csv(
    "../datasets/mental_health_burnout_prediction_dataset.csv"
)

df.head()

,Person_ID,Age,Gender,Country,Occupation,Education_Level,Employment_Status,Monthly_Income_USD,Work_Hours_Per_Week,Remote_Work,...,Family_History_Mental_Illness,Therapy_Attendance,Support_System,Life_Satisfaction,Productivity_Score,Absenteeism_Days,Burnout_Score,Mental_Health_Status,AI_Wellness_Recommendation,Burnout_Risk
0,1,58.0,Male,United States,Marketing Manager,Diploma,Employed,2786.0,67,Hybrid,...,No,Yes,Average,6.0,19.0,2,82,Critical,Practice Mindfulness,High
1,2,24.0,Female,Pakistan,Freelancer,Bachelor,Student,8027.0,44,Hybrid,...,Yes,No,Good,4.0,27.0,22,64,Critical,Reduce Workload,Moderate
2,3,59.0,Male,Malaysia,Accountant,PhD,Employed,3177.0,60,No,...,Yes,No,Average,4.0,51.0,17,50,Needs Attention,Increase Physical Activity,Moderate
3,4,65.0,Other,Mexico,Chef,Master,Self-Employed,4093.0,25,Yes,...,No,Yes,Good,7.0,83.0,14,16,Healthy,Maintain Healthy Routine,Low
4,5,64.0,Other,Japan,Farmer,Diploma,Student,NaN,52,No,...,Yes,No,Poor,9.0,61.0,4,26,Healthy,Reduce Workload,Low


# Criação do Dataset para Pré-processamento

In [2]:
df_clean = df.copy()

print("Cópia criada com sucesso.")
print("Dimensões:", df_clean.shape)

Cópia criada com sucesso.
Dimensões: (50000, 40)


# Tratamento de Valores Ausentes

In [ ]:
numericas = df_clean.select_dtypes(
    include=np.number
).columns

numericas_sem_id = [
    coluna for coluna in numericas
    if coluna != "Person_ID"
]

for coluna in numericas_sem_id:
    df_clean[coluna] = df_clean[coluna].fillna(
        df_clean[coluna].median()
    )

Tratamento das Variáveis Categóricas:

In [4]:
categoricas = df_clean.select_dtypes(
    include="object"
).columns

for coluna in categoricas:
    df_clean[coluna] = df_clean[coluna].fillna(
        df_clean[coluna].mode()[0]
    )

Resultado:

In [ ]:
total_ausentes = df_clean.isnull().sum().sum()

print("Total de valores ausentes após o tratamento:", total_ausentes)

0


# Análise e Tratamento de Outliers

A análise realizada pelo método IQR no notebook de EDA não identificou outliers nas variáveis numéricas. Dessa forma, não serão removidas observações, evitando a exclusão de valores potencialmente válidos.

# Codificação das Variáveis Categóricas

Para as variáveis categóricas utilizadas como características dos modelos, será utilizado One-Hot Encoding, pois a maioria das variáveis representa categorias nominais sem uma ordem numérica natural.

Label Encoding será considerado principalmente para variáveis ordinais ou para a representação do alvo de classificação, quando necessário.

In [15]:
categoricas = df_clean.select_dtypes(
    include="object"
).columns

print("Variáveis categóricas:")
for coluna in categoricas:
    print("-", coluna)

Variáveis categóricas:
- Gender
- Country
- Occupation
- Education_Level
- Employment_Status
- Remote_Work
- Sleep_Quality
- Stress_Level
- Exercise_Frequency
- Alcohol_Consumption
- Smoking
- Healthy_Diet
- Chronic_Stress
- Family_History_Mental_Illness
- Therapy_Attendance
- Support_System
- Mental_Health_Status
- AI_Wellness_Recommendation
- Burnout_Risk


## One-Hot Encoding

O One-Hot Encoding transforma cada categoria em uma variável binária. Essa técnica é adequada para variáveis nominais, evitando atribuir uma ordem numérica artificial às categorias.

In [16]:
df_encoded = pd.get_dummies(
    df_clean,
    columns=categoricas,
    drop_first=False,
    dtype=int
)

print("Dimensões após One-Hot Encoding:", df_encoded.shape)

df_encoded.head()

Dimensões após One-Hot Encoding: (50000, 121)


,Person_ID,Age,Monthly_Income_USD,Work_Hours_Per_Week,Job_Satisfaction,Work_Life_Balance,Sleep_Hours,Anxiety_Score,Depression_Score,Mood_Score,...,AI_Wellness_Recommendation_Improve Sleep,AI_Wellness_Recommendation_Increase Physical Activity,AI_Wellness_Recommendation_Maintain Healthy Routine,AI_Wellness_Recommendation_Practice Mindfulness,AI_Wellness_Recommendation_Reduce Workload,AI_Wellness_Recommendation_Seek Professional Support,AI_Wellness_Recommendation_Take Regular Breaks,Burnout_Risk_High,Burnout_Risk_Low,Burnout_Risk_Moderate
0,1,58.0,2786.0,67,3.0,7.0,4.6,74.0,75.0,26.0,...,0,0,0,1,0,0,0,1,0,0
1,2,24.0,8027.0,44,5.0,8.0,4.7,58.0,80.0,29.0,...,0,0,0,0,1,0,0,0,0,1
2,3,59.0,3177.0,60,8.0,7.0,6.3,54.0,55.0,44.0,...,0,1,0,0,0,0,0,0,0,1
3,4,65.0,4093.0,25,2.0,9.0,7.1,12.0,11.0,88.0,...,0,0,1,0,0,0,0,0,1,0
4,5,64.0,6231.0,52,1.0,4.0,9.1,25.0,48.0,67.0,...,0,0,0,0,1,0,0,0,1,0


# Padronização das Variáveis Numéricas

A padronização será aplicada às variáveis numéricas utilizadas como características dos modelos. O identificador Person_ID não será padronizado, pois não representa uma característica preditiva.

In [17]:
numericas_modelo = [
    coluna
    for coluna in df_clean.select_dtypes(include=np.number).columns
    if coluna != "Person_ID"
]

scaler = StandardScaler()

df_encoded[numericas_modelo] = scaler.fit_transform(
    df_encoded[numericas_modelo]
)

print("Padronização concluída.")

Padronização concluída.


# Salvar dataset limpo

In [8]:
df_clean.to_csv(
    "../datasets/dataset_clean.csv",
    index=False
)

# Preparação do Dataset para Regressão

In [ ]:
# Variável-alvo da regressão
target_reg = "Burnout_Score"

# Variáveis que não devem ser utilizadas como preditoras
colunas_excluir_reg = [
    "Burnout_Score",
    "Burnout_Risk",
    "AI_Wellness_Recommendation",
    "Person_ID"
]

X_reg = df_encoded.drop(
    columns=colunas_excluir_reg,
    errors="ignore"
)

y_reg = df_clean[target_reg]

# Montagem do dataset final
dataset_regression = X_reg.copy()
dataset_regression[target_reg] = y_reg

#Salvar o dataset de regressão em CSV
dataset_regression.to_csv(
    "../datasets/dataset_regression.csv",
    index=False
)

print(
    "Dataset de regressão salvo com sucesso:",
    dataset_regression.shape
)

# Preparação do Dataset para Classificação

In [ ]:
# Variável-alvo da classificação
target_class = "Burnout_Risk"

# Variáveis excluídas para evitar vazamento de informação
colunas_excluir_class = [
    "Burnout_Risk",
    "Burnout_Score",
    "Mental_Health_Status",
    "AI_Wellness_Recommendation",
    "Person_ID"
]

X_class = df_encoded.drop(
    columns=colunas_excluir_class,
    errors="ignore"
)

y_class = df_clean[target_class]

dataset_classification = X_class.copy()
dataset_classification[target_class] = y_class

# Salvar o dataset de classificação em CSV
dataset_classification.to_csv(
    "../datasets/dataset_classification.csv",
    index=False
)

print(
    "Dataset de classificação salvo com sucesso:",
    dataset_classification.shape
)

# Considerações sobre o Pré-processamento

Os valores ausentes numéricos foram preenchidos pela mediana e os categóricos pela moda. A análise de outliers pelo método IQR não indicou necessidade de remoção. As variáveis categóricas utilizadas como características foram codificadas por One-Hot Encoding. A padronização foi aplicada às variáveis numéricas para adequação a algoritmos sensíveis à escala.